## We can create ease in accessing information from newspapers through web scraping. Here we use popular web scraping tools in Python to generate a CSV file containing requisite information from different sections of the web edition of *Indian Express*, a popular national daily in India. 

## This notebook can be scheduled to run daily to access info on a regular basis. While news websites can often distract us through information overload, seeking information like this can help us save time and even organize our news sources in a methodical manner. 

### Importing necessary packages

In [ ]:
import pandas as pd
import os
from datetime import datetime 
from datetime import timedelta
from datetime import date
from array import *
import pdb

### Getting articles from UPSC Essentials page

#### A page highlighting the most important news sources for the UPSC examination, one of the toughest exams in the world, selecting an elite cohort of officers to the higher echelons of the Indian bureaucracy

#### To scrape the articles, we will be using the popular Python package of BeautifulSoup 

In [ ]:
%%capture
!pip install lxml
!pip install html5lib
!pip install requests
!pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup as bs
import requests
from urllib.request import Request, urlopen

In [ ]:
url = "https://indianexpress.com/section/upsc-current-affairs/"      #URL of the requisite webpage containing UPSC news sources
text = requests.get(url)
soup = bs(text.content, 'html.parser')                               #Parsing text content for extraction

In [ ]:
#soup.prettify()

In [ ]:
sec_cont = []
sec_cont = soup.find('div', {'class':'nation'})                    #isolating the required section within the parse text

In [ ]:
#sec_cont

#### We will be using the following method to scrape text from each section:
##### 1. Identify and list links of news articles in the body of the webpage
##### 2. Use links to source the articles
##### 3. Elicit requisite information from each article like Title, Date, Subheadings etc. (from the web elements of the page)
##### 4. Sort them by dates and filter the articles of the day.
##### 5. Create a dataframe to store the information. 

In [ ]:
upsc_ca_premium_list = []
links=[]

for link in sec_cont.findAll('a', href=True):                         #function to automate scraping of each article in the list
    
    upsc_ca = {}
    page_url = link.get('href')
    
    try:

        # URL

        upsc_ca['URL'] = page_url

        # Invoke URL

        page = requests.get(page_url)
        page_soup = bs(page.content, 'lxml')

        # Title

        upsc_ca['Title'] = page_soup.find('title').text

        # Content

        page_content = ''
        page_soup_div = page_soup.find_all('h2', {'class': 'synopsis'})
        for p_content in page_soup_div:
            page_content = page_content + p_content.text

        # Content

        upsc_ca['Content'] = page_content

        # Date Time

        page_soup_span = page_soup.find_all('span',{'itemprop': 'dateModified'})
        upsc_ca['Publish Date'] = page_soup_span[0].text
        
        links.append(link.get('href'))
        upsc_ca_premium_list.append(upsc_ca)
    
    except:
        print ('ERROR!', page_url)
        
print ('Extracted', len(upsc_ca_premium_list), 'articles from UPSC Current Affairs')
print ('Done')

In [ ]:
upsc_ca_premium_list = pd.DataFrame.from_dict(upsc_ca_premium_list)

In [ ]:
upsc_ca = upsc_ca_premium_list.copy()

In [ ]:
upsc_ca['URL']= upsc_ca['URL'].astype('string')
upsc_ca['Title']= upsc_ca['Title'].astype('string')
upsc_ca['Content']= upsc_ca['Content'].astype('string')
upsc_ca['Publish Date']= upsc_ca['Publish Date'].astype('string')

upsc_ca = upsc_ca.iloc[::2]    #Dropping duplicate rows

upsc_ca['Publish Date'] = upsc_ca['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
upsc_ca['Publish Date'] = pd.to_datetime(upsc_ca['Publish Date']).dt.strftime("%Y-%m-%d")

today = date.today()
yesterday = today - timedelta(days = 1)
yesterday = yesterday.strftime("%Y-%m-%d")

try:
    
    upsc_current_affairs = upsc_ca.loc[upsc_ca['Publish Date']==today]
    
except:
    print("No UPSC Current Affairs today")

In [ ]:
upsc_current_affairs

# Getting articles from Opinion page

In [ ]:
url = "https://indianexpress.com/section/opinion/"
opeds = requests.get(url)
soup = bs(opeds.content, 'html.parser')

soup.prettify

op_cont = []
op_cont = soup.find('div', {'class':'o-opin'})

oped_list = []
articles=[]

for article in op_cont.findAll('a', href=True):
    
    opinion = {}
    op_url = article.get('href')
    
    try:

        # URL

        opinion['URL'] = op_url

        # Invoke URL

        page = requests.get(op_url)
        page_soup = bs(page.content, 'html.parser')

        # Title

        page_soup_title = page_soup.find('h1', {'class':'native_story_title'})
        opinion['Title']= page_soup_title.text

        # Content

        page_content = ''
        page_soup_div = page_soup.find_all('h2',{"class":"synopsis"})
        for p_content in page_soup_div:
            page_content = page_content + p_content.text

        # Content

        opinion['Content'] = page_content 

        # Date Time
        
        dates=[]
        dates = page_soup.find('div',{'id':'storycenterbyline'})
        for date in dates.select_one('span'):
            opinion['Publish Date'] = date.text
        
        articles.append(article.get('href'))
        oped_list.append(opinion)

    
    except Exception as e:
        print(e, op_url)

print ('Extracted', len(oped_list),'opinion articles')
print ('Done with extracting all op-eds')

In [ ]:
opinion_list = pd.DataFrame.from_dict(oped_list)
opinion_list

opinion_list.info()

opinion_list['URL']= opinion_list['URL'].astype('string')
opinion_list['Title']= opinion_list['Title'].astype('string')
opinion_list['Content']= opinion_list['Content'].astype('string')
opinion_list['Publish Date']= opinion_list['Publish Date'].astype('string')

opinion_list = opinion_list.iloc[::2]    #Dropping duplicate rows


opinion_list['Publish Date'] = opinion_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
opinion_list['Publish Date'] = pd.to_datetime(opinion_list['Publish Date']).dt.strftime("%Y-%m-%d")

In [ ]:
from datetime import date
today = date.today().strftime("%Y-%m-%d")
today

try:
    op_columns = opinion_list.loc[opinion_list['Publish Date']==today]
except:
    op_columns=[]
    print("No Opinion Columns today")

In [ ]:
op_columns

# Getting articles from Explained pages

In [ ]:
def explained(x):
    url = (("https://indianexpress.com/about/explained-") + (x))
    expl_cont = requests.get(url)
    soup = bs(expl_cont.content, 'html.parser')

    soup.prettify

    sec_exp = []
    sec_exp = soup.find('div', {'class':'search-result'})

    links =[]

    for link in sec_exp.findAll('a', href=True):
        section_explained = {}
        sec_exp_url = link.get('href')
    
        try:
            # URL

            section_explained['URL'] = sec_exp_url

            # Invoke URL

            page = requests.get(sec_exp_url)
            page_soup = bs(page.content, 'html.parser')

            # Title

            page_soup_title = page_soup.find('h1', {'class':'native_story_title'})
            section_explained['Title']= page_soup_title.text

            # Content

            page_content = ''
            page_soup_div = page_soup.find_all('h2',{"class":"synopsis"})
            for p_content in page_soup_div:
                page_content = page_content + p_content.text

            # Content

            section_explained['Content'] = page_content 

            # Date Time

            dates=[]
            dates = page_soup.find('div',{'id':'storycenterbyline'})
            for date in dates.select_one('span'):
                section_explained['Publish Date'] = date.text

            links.append(link.get('href'))
            sec_exp_list.append(section_explained)

        except Exception as e:
            print(e, sec_exp_url)

    print ('Extracted', len(sec_exp_list), 'explained articles in ' + (x))
    print ('Done with extracting all explained articles in ' + (x))

### Economics

In [ ]:
sec_exp_list = []
explained('economics')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    econ_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    econ_explained=[]
    print("No Explained Economics today")
    
econ_explained

### Health

In [ ]:
sec_exp_list = []
explained('health')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    health_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    health_explained=[]
    print("No Explained Health today")
    
health_explained

### Politics

In [ ]:
sec_exp_list = []
explained('politics')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    politics_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    politics_explained=[]
    print("No Explained Politics today")
    
politics_explained

### Culture

In [ ]:
sec_exp_list = []
explained('culture')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    culture_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    culture_explained=[]
    print("No Explained Culture today")
    
culture_explained

### Global

In [ ]:
sec_exp_list = []
explained('global')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    global_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==yesterday]
except:
    global_explained=[]
    print("No Explained Global today")
    
global_explained

### Sci-tech

In [ ]:
sec_exp_list = []
explained('sci-tech')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    scitech_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    scitech_explained=[]
    print("No Explained Sci-Tech today")
    
scitech_explained

### Climate

In [ ]:
sec_exp_list = []
explained('climate')

In [ ]:
sec_exp_list = pd.DataFrame.from_dict(sec_exp_list)
sec_exp_list

sec_exp_list.info()

sec_exp_list['URL']= sec_exp_list['URL'].astype('string')
sec_exp_list['Title']= sec_exp_list['Title'].astype('string')
sec_exp_list['Content']= sec_exp_list['Content'].astype('string')
sec_exp_list['Publish Date']= sec_exp_list['Publish Date'].astype('string')


sec_exp_list = sec_exp_list.iloc[::2]    #Dropping duplicate rows
sec_exp_list

sec_exp_list['Publish Date'] = sec_exp_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))
sec_exp_list['Publish Date']

sec_exp_list['Publish Date'] = pd.to_datetime(sec_exp_list['Publish Date']).dt.strftime("%Y-%m-%d")
sec_exp_list['Publish Date']

try:
    climate_explained = sec_exp_list.loc[sec_exp_list['Publish Date']==today]
except:
    climate_explained=[]
    print("No Explained Climate today")
    
climate_explained

In [ ]:
explained_list = [econ_explained,health_explained,politics_explained, culture_explained, global_explained, scitech_explained, climate_explained]

In [ ]:
explained = pd.concat(explained_list)
explained

# Science section

In [ ]:
url = "https://indianexpress.com/section/science/"
markup = requests.get(url)
soup = bs(markup.content, 'html.parser')

soup.prettify

sci_cont = []
sci_cont = soup.find('div', {'class':'nation'})

science_list = []
links=[]

for link in sci_cont.findAll('a', href=True):
    
    science = {}
    sci_url = link.get('href')
    
    try:

        # URL

        science['URL'] = sci_url

        # Invoke URL

        page = requests.get(sci_url)
        page_soup = bs(page.content, 'html.parser')

        # Title

        page_soup_title = page_soup.find('h1', {'class':'native_story_title'})
        science['Title']= page_soup_title.text

        # Content

        page_content = ''
        page_soup_div = page_soup.find_all('h2',{"class":"synopsis"})
        for p_content in page_soup_div:
            page_content = page_content + p_content.text

        # Content

        science['Content'] = page_content 

        # Date Time
        
        dates=[]
        dates = page_soup.find('div',{'id':'storycenterbyline'})
        for date in dates.select_one('span'):
            science['Publish Date'] = date.text
        
        links.append(link.get('href'))
        science_list.append(science)
    
    except Exception as e:
        print(e, sci_url)
        
print ('Extracted', len(science_list), 'science articles')
print ('Done with extracting all science articles')

In [ ]:
science = pd.DataFrame.from_dict(science_list)
science

science.info()

science['URL']= science['URL'].astype('string')
science['Title']= science['Title'].astype('string')
science['Content']= science['Content'].astype('string')
science['Publish Date']= science['Publish Date'].astype('string')

science = science.iloc[::2]    #Dropping duplicate rows


science['Publish Date'] = science['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))

science['Publish Date'] = pd.to_datetime(science['Publish Date']).dt.strftime("%Y-%m-%d")
science

In [ ]:
from datetime import date
today = date.today().strftime("%Y-%m-%d")
today

try:
    science = science.loc[science['Publish Date']==today]
except:
    science=[]
    print("No Science Columns today")

In [ ]:
science

# Business section: Economy + Banking and Finance

In [ ]:
biz_list=[]
def biz(x):
    url = (("https://indianexpress.com/section/business/") + (x))
    print(url)
    business_cont = requests.get(url)
    soup = bs(business_cont.content, 'html.parser')

    soup.prettify

    business_list=[]
    business_list = soup.find('div', {'class':'nation'})

    links =[]

    for link in business_list.find_all('a', href=True):
        
        business_page = {}
        business_url = link.get('href')
    
        try:
            # URL

            business_page['URL'] = business_url

            # Invoke URL

            page = requests.get(business_url)
            page_soup = bs(page.content, 'html.parser')

            # Title

            page_soup_title = page_soup.find('h1', {'class':'native_story_title'})
            business_page['Title']= page_soup_title.text

            # Content

            page_content = ''
            page_soup_div = page_soup.find_all('h2',{"class":"synopsis"})
            for p_content in page_soup_div:
                page_content = page_content + p_content.text

            # Content

            business_page['Content'] = page_content 

            # Date Time

            dates=[]
            dates = page_soup.find('div',{'id':'storycenterbyline'})
            for date in dates.select_one('span'):
                business_page['Publish Date'] = date.text

            links.append(link.get('href'))
            biz_list.append(business_page)


        except Exception as e:
            print(e, business_url)
            
    print ("Extracted", len(biz_list), "pages in " + (x))
    print ('Done with extracting all business articles in ' + (x))

In [ ]:
biz_list=[]
biz('economy')

In [ ]:
biz_list = pd.DataFrame.from_dict(biz_list)
biz_list

biz_list.info()

biz_list['URL']= biz_list['URL'].astype('string')
biz_list['Title']= biz_list['Title'].astype('string')
sec_exp_list['Content']= biz_list['Content'].astype('string')
biz_list['Publish Date']= biz_list['Publish Date'].astype('string')


biz_list = biz_list.iloc[::2]    #Dropping duplicate rows

biz_list['Publish Date'] = biz_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))

biz_list['Publish Date'] = pd.to_datetime(biz_list['Publish Date']).dt.strftime("%Y-%m-%d")

try:
    economy = biz_list.loc[biz_list['Publish Date']==today]
except:
    economy=[]
    print("No Business:Economy today")
    
economy

In [ ]:
biz_list=[]
biz('banking-and-finance')

In [ ]:
biz_list = pd.DataFrame.from_dict(biz_list)
biz_list

biz_list.info()

biz_list['URL']= biz_list['URL'].astype('string')
biz_list['Title']= biz_list['Title'].astype('string')
sec_exp_list['Content']= biz_list['Content'].astype('string')
biz_list['Publish Date']= biz_list['Publish Date'].astype('string')


biz_list = biz_list.iloc[::2]    #Dropping duplicate rows

biz_list['Publish Date'] = biz_list['Publish Date'].map(lambda x: x.lstrip('Updated: '.rstrip('')))

biz_list['Publish Date'] = pd.to_datetime(biz_list['Publish Date']).dt.strftime("%Y-%m-%d")

try:
    bfsi = biz_list.loc[biz_list['Publish Date']==today]
except:
    bfsi=[]
    print("No Business:BFSI today")
    
bfsi

In [ ]:
read_list = [upsc_current_affairs,op_columns,explained,science,economy,bfsi]

In [ ]:
bfsi.info()

In [ ]:
read = pd.concat(read_list)
read.reset_index(inplace = True, drop = True)

In [ ]:
read = read[['Publish Date','Title','Content','URL']]
read

In [ ]:
today = date.today()
yesterday = today - timedelta(days = 1)

yesterday = yesterday.strftime("%b-%d-%Y")
today = today.strftime("%b-%d-%Y")

In [ ]:
read.to_csv(today + ' IE extracts.csv', index=False)

In [ ]:
print("File created for daily IE reading")